**<h1><center>Homework 1</center></h1>**
<center>Course: Machine Learning For Signal Processing</center>
<center>Amirkabir University of Technology (Tehran Polytechnic) - Department of Electrical Engineering
</center>

---

Module 1 assignment.   
Deadline: 1405/08/14, 11:59pm. 
___

Student's Name: [YOUR NAME]  
Student's ID: [YOUR ID]  


*If you have any questions, please contact the TAs.* 
___

# 0. Setup and shared utilities

Run this section before starting any question. All cells in Section 0 are provided

## 0.1 Environment and imports

In [ ]:
# PROVIDED | Environment and imports

import sys
from pathlib import Path

import numpy as np
import pandas as pd

import scipy
from scipy import linalg, optimize, special, stats

import matplotlib
import matplotlib.pyplot as plt

import sklearn
from sklearn import metrics
from sklearn.metrics import adjusted_rand_score
from sklearn.linear_model import Lasso
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import (
    PolynomialFeatures,
    SplineTransformer,
    StandardScaler,
)

versions = {
    "Python": sys.version.split()[0],
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
    "scikit-learn": sklearn.__version__,
}

print(pd.Series(versions, name="version").to_string())

## 0.2 Configuration and reproducibility

Use the supplied configurations for required experiments.

Create a random generator once per experiment. Recreating the same named stream and replicate reproduces its draws; changing the replicate creates a different reproducible stream.

In [ ]:
# PROVIDED | Reproducible random streams and shared configurations

BASE_SEED = 2026

STREAM_IDS = {
    "q1_observation": 10,
    "q1_monte_carlo": 20,
    "q1_noise_sweep": 30,
    "q1_colored_noise": 40,
    "q1_prior": 50,
    "q1_discrete": 60,
    "q1_detection_calibration": 70,
    "q1_detection_evaluation": 80,
    "q2_split": 100,
    "q2_resampling": 110,
}


def make_rng(stream_name, replicate=0):
    if stream_name not in STREAM_IDS:
        raise KeyError(f"Unknown random stream: {stream_name}")

    if not isinstance(replicate, (int, np.integer)) or replicate < 0:
        raise ValueError("replicate must be a nonnegative integer")

    seed = np.random.SeedSequence(
        [BASE_SEED, STREAM_IDS[stream_name], int(replicate)]
    )

    return np.random.default_rng(seed)


# Initial generated-signal experiment. later tasks add local configurations.
Q1_CONFIG = {
    "n_samples": 121,
    "pulse_onset": -1.5,
    "rise_time": 0.20,
    "decay_time": 1.0,
    "true_amplitude": 1.5,
    "noise_sd": 0.48,
    "n_trials": 4000,
    "noise_sd_grid": np.array([0.12, 0.24, 0.48, 0.96]),
    "target_false_alarm": 0.05,
}

# Split fractions apply to groups. Please set the data path accordingly
Q2_CONFIG = {
    "data_path": Path("data") / "airfoil_self_noise.dat",
    "test_fraction": 0.20,
    "validation_fraction_of_remaining": 0.25,
    "cv_folds": 3,
}

print("Configurations and named random streams are ready")

## 0.3 Shared plotting style

Use consistent colors and labeled axes throughout the homework.

Question 1 includes supplied plotting cells. In Questions 2 and 3,
create the required visualizations using this shared style.

In [ ]:
# PROVIDED | Shared plotting style

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 180,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
    "axes.axisbelow": True,
    "legend.frameon": False,
})

COLORS = {
    "truth": "#263447",
    "mle": "#007F82",
    "peak": "#BC5C3D",
    "map": "#7356A0",
    "mmse": "#B38318",
    "data": "#AFB8C3",
    "train": "#007F82",
    "validation": "#BC5C3D",
    "test": "#263447",
}


def finish_figure(fig, title=None):
    if title is None:
        fig.tight_layout()
    else:
        fig.suptitle(title, fontsize=12)
        fig.tight_layout(rect=(0, 0, 1, 0.95))

    plt.show()

## 0.4 Shared validation and evaluation helpers

These utilities check inputs, summarize results, and audit data partitions. They do not fit estimators, select thresholds, or construct data splits.

Undefined performance ratios are returned as `np.nan`.

In [ ]:
# PROVIDED | Numerical validation and scalar-estimator summaries

def as_float_array(values, name="array", ndim=None):
    if np.iscomplexobj(values):
        raise ValueError(f"{name} must be real-valued")

    array = np.asarray(values, dtype=float)

    if ndim is not None and array.ndim != ndim:
        raise ValueError(f"{name} must have {ndim} dimensions")

    if array.size == 0 or not np.isfinite(array).all():
        raise ValueError(f"{name} must be nonempty and finite")

    return array


def validate_spd(matrix, name="covariance"):
    matrix = as_float_array(matrix, name, ndim=2)

    if matrix.shape[0] != matrix.shape[1]:
        raise ValueError(f"{name} must be square")

    if not np.allclose(matrix, matrix.T, rtol=1e-10, atol=1e-12):
        raise ValueError(f"{name} must be symmetric")

    try:
        linalg.cholesky(matrix, lower=True)
    except linalg.LinAlgError as exc:
        raise ValueError(f"{name} must be positive definite") from exc

    return matrix


def summarize_estimates(estimates, truth):
    estimates = as_float_array(estimates, "estimates", ndim=1)
    truth = float(as_float_array(truth, "truth", ndim=0))

    if estimates.size < 2:
        raise ValueError("At least two estimates are required")

    return {
        "bias": float(estimates.mean() - truth),
        "variance": float(estimates.var(ddof=1)),
        "mse": float(np.mean((estimates - truth)**2)),
    }

In [ ]:
# PROVIDED | Binary-decision evaluation

def binary_summary(labels, scores, threshold):
    """Evaluate the rule: decide class 1 when score >= threshold.

    labels and scores must have shape (n_samples,).
    Larger scores must indicate stronger evidence for class 1.
    Infinite thresholds are allowed; NaN thresholds are not.
    """
    labels = as_float_array(labels, "labels", ndim=1)
    scores = as_float_array(scores, "scores", ndim=1)

    if labels.shape != scores.shape or not np.isin(labels, [0, 1]).all():
        raise ValueError("labels must be 0/1 and match the scores shape")

    if np.asarray(threshold).ndim != 0 or np.isnan(float(threshold)):
        raise ValueError("threshold must be a scalar other than NaN")

    predictions = (scores >= float(threshold)).astype(int)

    tn, fp, fn, tp = metrics.confusion_matrix(
        labels.astype(int),
        predictions,
        labels=[0, 1],
    ).ravel()

    def ratio(numerator, denominator):
        return float(numerator / denominator) if denominator else np.nan

    return {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "p_fa": ratio(fp, fp + tn),
        "p_d": ratio(tp, tp + fn),
        "precision": ratio(tp, tp + fp),
        "accuracy": ratio(tp + tn, labels.size),
    }

In [ ]:
# PROVIDED | Partition and group-integrity checks

def check_split_integrity(partitions, n_samples, groups=None):
    """Check a complete train/validation/test partition

    partitions: dictionary of one-dimensional integer index arrays
    groups: optional group identifier for every sample

    Returns sample counts. Raises ValueError for invalid partitions.
    This checks structural separation; it does not prove independence
    or rule out every form of leakage.
    """
    if set(partitions) != {"train", "validation", "test"}:
        raise ValueError("Expected train, validation, and test partitions")

    if not isinstance(n_samples, (int, np.integer)) or n_samples < 3:
        raise ValueError("n_samples must be an integer >= 3")

    checked = {}

    for name, indices in partitions.items():
        indices = np.asarray(indices)

        if (
            indices.ndim != 1
            or indices.size == 0
            or not np.issubdtype(indices.dtype, np.integer)
        ):
            raise ValueError(
                f"{name} indices must be a nonempty integer vector"
            )

        if np.any(indices < 0) or np.any(indices >= n_samples):
            raise ValueError(f"{name} contains an out-of-range index")

        if np.unique(indices).size != indices.size:
            raise ValueError(f"{name} contains duplicate indices")

        checked[name] = indices

    combined = np.concatenate(list(checked.values()))

    if (
        combined.size != n_samples
        or np.unique(combined).size != n_samples
    ):
        raise ValueError(
            "Partitions must be disjoint and cover all samples"
        )

    if groups is not None:
        groups = np.asarray(groups)

        if groups.shape != (n_samples,) or pd.isna(groups).any():
            raise ValueError(
                "groups must contain one nonmissing identifier per sample"
            )

        owners = {}

        for name, indices in checked.items():
            for group in set(groups[indices].tolist()):
                if group in owners:
                    raise ValueError(
                        f"Group {group!r} appears in multiple partitions"
                    )

                owners[group] = name

    return pd.Series(
        {name: len(indices) for name, indices in checked.items()},
        name="samples",
    )

## 0.5 Setup checks

Run the checks below. They use small artificial inputs and do not load, fit, or split the real dataset.

In [ ]:
# PROVIDED | Basic checks for the shared utilities

# Identical stream names and replicate indices reproduce identical draws
np.testing.assert_array_equal(
    make_rng("q1_observation").normal(size=5),
    make_rng("q1_observation").normal(size=5),
)

# Different replicate indices produce different draws
assert not np.array_equal(
    make_rng("q1_observation", replicate=0).normal(size=5),
    make_rng("q1_observation", replicate=1).normal(size=5),
)

# A valid covariance matrix is accepted
validate_spd(
    np.array([
        [1.0, 0.3],
        [0.3, 1.0],
    ])
)

# Check the conventions used by the estimator summary
check_stats = summarize_estimates(
    np.array([1.0, 2.0, 3.0]),
    truth=2.0,
)

np.testing.assert_allclose(
    [
        check_stats["bias"],
        check_stats["variance"],
        check_stats["mse"],
    ],
    [0.0, 1.0, 2.0 / 3.0],
)

# Check the confusion-matrix ordering and decision convention
check_binary = binary_summary(
    labels=[0, 0, 1, 1],
    scores=[0.1, 0.8, 0.4, 0.9],
    threshold=0.5,
)

assert [
    check_binary[key] for key in ("tn", "fp", "fn", "tp")
] == [1, 1, 1, 1]

# Check a small partition with disjoint groups
check_counts = check_split_integrity(
    partitions={
        "train": np.array([0, 1]),
        "validation": np.array([2, 3]),
        "test": np.array([4, 5]),
    },
    n_samples=6,
    groups=np.array([0, 0, 1, 1, 2, 2]),
)

assert check_counts.sum() == 6

print("Section 0 checks passed")
print("No dataset has been loaded or split")

# 1. Inference from generated signals

Helpful links:

1_ [Estimation Theory](https://www.epfl.ch/labs/la/wp-content/uploads/2018/08/EstimationTheory.pdf)

## 1.1 Warm-up: amplitude estimation and reliability

We observe a known waveform with an unknown amplitude:

$$
\mathbf{y}=a\mathbf{s}+\mathbf{n},
\qquad
\mathbf{n}\sim\mathcal{N}(\mathbf{0},\sigma^2\mathbf{I}).
$$

The amplitude is an unconstrained real parameter. Implement its maximum likelihood estimator, simulate repeated observations, and compare empirical performance with theory.

In [ ]:
# PROVIDED | Task 1.1 | Signal model and one noisy observation

warmup_t = np.linspace(-3.5, 3.5, Q1_CONFIG["n_samples"])

if not 0 < Q1_CONFIG["rise_time"] < Q1_CONFIG["decay_time"]:
    raise ValueError("Require 0 < rise_time < decay_time")

warmup_u = np.maximum(
    warmup_t - Q1_CONFIG["pulse_onset"],
    0.0,
)

warmup_s = (
    np.exp(-warmup_u / Q1_CONFIG["decay_time"])
    - np.exp(-warmup_u / Q1_CONFIG["rise_time"])
)

warmup_s = warmup_s / warmup_s.max()

warmup_a_true = Q1_CONFIG["true_amplitude"]
warmup_sigma = Q1_CONFIG["noise_sd"]

warmup_clean = warmup_a_true * warmup_s

warmup_rng = make_rng("q1_observation")
warmup_y = warmup_clean + warmup_rng.normal(
    loc=0.0,
    scale=warmup_sigma,
    size=warmup_s.size,
)


def estimate_amplitude_peak(observations, template):
    observations = np.asarray(observations, dtype=float)
    template = np.asarray(template, dtype=float)

    peak_index = np.argmax(np.abs(template))

    return observations[..., peak_index] / template[peak_index]

### 1.1a Implement maximum likelihood estimation

Complete `estimate_amplitude_mle`. For this warm-up, the familiar result is supplied:

$$
\widehat a_{\mathrm{MLE}} =
\frac{\mathbf{s}^{T}\mathbf{y}}{\mathbf{s}^{T}\mathbf{s}}.
$$

Support both one observation and a batch with one observation per row.
Use NumPy operations; do not call an optimizer or a regression fitting library.

In [ ]:
# TODO-FUNCTION | Task 1.1a | Maximum likelihood amplitude estimator

def estimate_amplitude_mle(observations, template):
    observations = as_float_array(observations, "observations")
    template = as_float_array(template, "template", ndim=1)

    if (
        observations.ndim not in (1, 2)
        or observations.shape[-1] != template.size
    ):
        raise ValueError(
            "observations must have shape (N,) or (K, N)"
        )

    if not np.any(template):
        raise ValueError("template must be nonzero")

    # TODO: compute and return the amplitude estimate(s).
    raise NotImplementedError("Complete Task 1.1a")

In [ ]:
# PROVIDED | Task 1.1a | Public checks and single-observation estimates

probe_s = np.array([0.2, -0.7, 1.1, 0.4])
probe_a = np.array([-1.2, 0.0, 2.3])
probe_y = probe_a[:, None] * probe_s[None, :]

# Noiseless recovery
np.testing.assert_allclose(
    estimate_amplitude_mle(probe_y, probe_s),
    probe_a,
    atol=1e-12,
)


assert np.ndim(
    estimate_amplitude_mle(probe_y[0], probe_s)
) == 0

np.testing.assert_allclose(
    estimate_amplitude_mle(probe_y, 3.0 * probe_s),
    probe_a / 3.0,
    atol=1e-12,
)

warmup_mle = float(
    estimate_amplitude_mle(warmup_y, warmup_s)
)
warmup_peak = float(
    estimate_amplitude_peak(warmup_y, warmup_s)
)

warmup_residual = warmup_y - warmup_mle * warmup_s

np.testing.assert_allclose(
    warmup_s @ warmup_residual,
    0.0,
    atol=1e-10,
)

print("MLE checks passed.")
print(f"True amplitude: {warmup_a_true:.4f}")
print(f"MLE estimate:  {warmup_mle:.4f}")
print(f"Peak estimate: {warmup_peak:.4f}")

In [ ]:
# PLOT/EVALUATION | Figure 1.1a | Observation and reconstructions

fig, axes = plt.subplots(
    1, 2,
    figsize=(11.8, 3.6),
    sharex=True,
    sharey=True,
)

axes[0].plot(
    warmup_t,
    warmup_y,
    color=COLORS["data"],
    lw=1,
    label="Observed waveform",
)
axes[0].plot(
    warmup_t,
    warmup_clean,
    color=COLORS["truth"],
    lw=2,
    label="Clean signal (simulation only)",
)

warmup_peak_index = np.argmax(np.abs(warmup_s))

axes[0].scatter(
    warmup_t[warmup_peak_index],
    warmup_y[warmup_peak_index],
    color=COLORS["peak"],
    s=25,
    zorder=4,
    label="Sample used by peak baseline",
)
axes[0].set_title("What was measured?")

axes[1].plot(
    warmup_t,
    warmup_clean,
    color=COLORS["truth"],
    lw=2,
    label="Clean signal",
)
axes[1].plot(
    warmup_t,
    warmup_mle * warmup_s,
    color=COLORS["mle"],
    lw=2,
    label="MLE reconstruction",
)
axes[1].plot(
    warmup_t,
    warmup_peak * warmup_s,
    color=COLORS["peak"],
    lw=1.8,
    ls="--",
    label="Peak reconstruction",
)
axes[1].set_title("What do the estimates reconstruct?")

for ax in axes:
    ax.set_xlabel("Sample coordinate")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Signal value")

finish_figure(fig)

### 1.1b Simulate repeated observations

Complete `run_monte_carlo` using independent white Gaussian noise for every trial and sample. Keep the true amplitude and template fixed.

Apply both estimators to the **same observations**. Reuse the estimator functions and the supplied random generator; do not reseed inside the function.

In [ ]:
# TODO-FUNCTION | Task 1.1b | Monte Carlo simulation

def run_monte_carlo(template, amplitude, noise_sd, trials, rng):

    template = as_float_array(template, "template", ndim=1)
    amplitude = float(
        as_float_array(amplitude, "amplitude", ndim=0)
    )
    noise_sd = float(
        as_float_array(noise_sd, "noise_sd", ndim=0)
    )

    if not np.any(template):
        raise ValueError("template must be nonzero")

    if (
        noise_sd <= 0
        or not isinstance(trials, (int, np.integer))
        or trials < 2
    ):
        raise ValueError(
            "noise_sd must be positive; trials must be an integer >= 2"
        )

    if not isinstance(rng, np.random.Generator):
        raise TypeError("rng must be a NumPy Generator")

    # TODO:
    # 1. Generate independent Gaussian noise for all trials and samples.
    # 2. Construct the observations.
    # 3. Apply both estimator functions to those same observations.
    # 4. Dictionary with keys "mle" and "peak". Each value must be an array of shape (trials,).
    raise NotImplementedError("Complete Task 1.1b")

In [ ]:
# PROVIDED | Task 1.1b | Interface and reproducibility checks

warmup_check_a = run_monte_carlo(
    warmup_s,
    warmup_a_true,
    warmup_sigma,
    trials=25,
    rng=make_rng("q1_monte_carlo", replicate=999),
)

warmup_check_b = run_monte_carlo(
    warmup_s,
    warmup_a_true,
    warmup_sigma,
    trials=25,
    rng=make_rng("q1_monte_carlo", replicate=999),
)

for method in ("mle", "peak"):
    assert np.asarray(warmup_check_a[method]).shape == (25,)
    assert np.isfinite(warmup_check_a[method]).all()
    assert np.std(warmup_check_a[method]) > 0

    np.testing.assert_array_equal(
        warmup_check_a[method],
        warmup_check_b[method],
    )

warmup_check_c = run_monte_carlo(
    warmup_s,
    warmup_a_true,
    warmup_sigma,
    trials=25,
    rng=make_rng("q1_monte_carlo", replicate=1000),
)

assert not np.array_equal(
    warmup_check_a["mle"],
    warmup_check_c["mle"],
)

print("Monte Carlo checks passed.")

### 1.1c Validate reliability against theory

Run 4,000 trials at the initial noise level, then repeat across the supplied noise standard deviations. Each noise level generates new observations.

Compare empirical bias, variance, and MSE with the supplied references:

$$
\operatorname{Var}(\widehat a_{\mathrm{MLE}}) =
\frac{\sigma^2}{\mathbf{s}^{T}\mathbf{s}} =
\mathrm{CRLB},
\qquad
\operatorname{Var}(\widehat a_{\mathrm{peak}}) =
\frac{\sigma^2}{s_{k_\star}^{2}}.
$$

Both estimators are unbiased under this model. Small empirical discrepancies are expected from finite Monte Carlo sampling.

In [ ]:
# EXPERIMENT | Task 1.1c | Fixed-noise experiment and actual-noise sweep

warmup_trials = Q1_CONFIG["n_trials"]
warmup_energy = float(warmup_s @ warmup_s)
warmup_peak_index = np.argmax(np.abs(warmup_s))

warmup_results = run_monte_carlo(
    warmup_s,
    warmup_a_true,
    warmup_sigma,
    warmup_trials,
    make_rng("q1_monte_carlo"),
)

warmup_fixed_rows = []

for method in ("mle", "peak"):
    summary = summarize_estimates(
        warmup_results[method],
        warmup_a_true,
    )

    theoretical_variance = warmup_sigma**2 / (
        warmup_energy
        if method == "mle"
        else warmup_s[warmup_peak_index]**2
    )

    warmup_fixed_rows.append({
        "method": method.upper(),
        **summary,
        "theory_variance": theoretical_variance,
    })

warmup_fixed_table = pd.DataFrame(warmup_fixed_rows)

warmup_sweep_rows = []

for replicate, sd in enumerate(Q1_CONFIG["noise_sd_grid"]):
    results = run_monte_carlo(
        warmup_s,
        warmup_a_true,
        float(sd),
        warmup_trials,
        make_rng("q1_noise_sweep", replicate=replicate),
    )

    warmup_sweep_rows.append({
        "noise_sd": float(sd),
        "noise_variance": float(sd**2),
        "mle_variance": summarize_estimates(
            results["mle"], warmup_a_true
        )["variance"],
        "crlb": float(sd**2 / warmup_energy),
        "peak_variance": summarize_estimates(
            results["peak"], warmup_a_true
        )["variance"],
    })

warmup_sweep_table = pd.DataFrame(warmup_sweep_rows)

print("Fixed-noise experiment:")
print(
    warmup_fixed_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\nActual-noise sweep:")
print(
    warmup_sweep_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

In [ ]:
# PLOT/EVALUATION | Figure 1.1c | Sampling uncertainty and variance

warmup_theory_var = {
    "mle": warmup_sigma**2 / warmup_energy,
    "peak": warmup_sigma**2 / warmup_s[warmup_peak_index]**2,
}

warmup_range_sd = np.sqrt(max(warmup_theory_var.values()))

warmup_grid = np.linspace(
    warmup_a_true - 4.5 * warmup_range_sd,
    warmup_a_true + 4.5 * warmup_range_sd,
    600,
)
warmup_bins = np.linspace(
    warmup_grid[0],
    warmup_grid[-1],
    90,
)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 3.8))

# Left: empirical sampling distributions and Gaussian references.
for method in ("peak", "mle"):
    axes[0].hist(
        warmup_results[method],
        bins=warmup_bins,
        density=True,
        alpha=0.28,
        color=COLORS[method],
        label=f"{method.upper()} trials",
    )

    axes[0].plot(
        warmup_grid,
        stats.norm.pdf(
            warmup_grid,
            loc=warmup_a_true,
            scale=np.sqrt(warmup_theory_var[method]),
        ),
        color=COLORS[method],
        lw=1.8,
        label=f"{method.upper()} theory",
    )

axes[0].axvline(
    warmup_a_true,
    color=COLORS["truth"],
    ls=":",
    label="True amplitude",
)
axes[0].set(
    xlabel="Estimated amplitude",
    ylabel="Probability density",
    title="Uncertainty across observations",
)

# Right: empirical variance versus actual noise variance.
warmup_noise_var = warmup_sweep_table["noise_variance"].to_numpy()

for method in ("mle", "peak"):
    reference = warmup_noise_var / (
        warmup_energy
        if method == "mle"
        else warmup_s[warmup_peak_index]**2
    )

    axes[1].loglog(
        warmup_noise_var,
        warmup_sweep_table[f"{method}_variance"],
        "o",
        color=COLORS[method],
        label=f"{method.upper()} empirical",
    )
    axes[1].loglog(
        warmup_noise_var,
        reference,
        "-",
        color=COLORS[method],
        lw=1.6,
        label=(
            "MLE theory / CRLB"
            if method == "mle"
            else "Peak theory"
        ),
    )

axes[1].set(
    xlabel="Actual noise variance",
    ylabel="Estimator variance",
    title="Does simulation agree with theory?",
)

for ax in axes:
    ax.legend(fontsize=7.5)

finish_figure(fig)

## 1.2 Colored noise and an unknown baseline

Now, the measurement includes an unknown constant baseline:
$$
\mathbf y=a\mathbf s+b\mathbf 1+\mathbf n
=\mathbf A\boldsymbol\theta+\mathbf n,
\qquad
\boldsymbol\theta=[a,b]^T.
$$

The noise covariance is known:
$$
[\boldsymbol\Sigma]_{ij}
=\sigma^2\rho^{|i-j|}, \qquad 0\leq\rho<1.
$$

Estimate both parameters and compare:
- amplitude estimation that ignores the baseline;
- ordinary least squares (OLS);
- correctly weighted Gaussian MLE, also called generalized least squares (GLS).


In [ ]:
# PROVIDED | Task 1.2 | Observation model and input validation

colored_s = warmup_s.copy()

colored_A = np.column_stack([
    colored_s,
    np.ones(colored_s.size),
])

colored_theta = np.array([
    Q1_CONFIG["true_amplitude"],
    0.35,  # unknown baseline
])

colored_sigma = Q1_CONFIG["noise_sd"]
colored_rho = 0.95
colored_trials = Q1_CONFIG["n_trials"]

colored_distance = np.abs(
    np.arange(colored_s.size)[:, None]
    - np.arange(colored_s.size)[None, :]
)


def make_colored_covariance(rho):
    if not np.isfinite(rho) or not 0 <= rho < 1:
        raise ValueError("rho must satisfy 0 <= rho < 1")

    return colored_sigma**2 * rho**colored_distance


def check_linear_inputs(observations, design, covariance):
    observations = as_float_array(observations, "observations")
    design = as_float_array(design, "design", ndim=2)
    covariance = validate_spd(covariance)

    if (
        observations.ndim not in (1, 2)
        or observations.shape[-1] != design.shape[0]
    ):
        raise ValueError(
            "observations and design have incompatible shapes"
        )

    if covariance.shape != (design.shape[0], design.shape[0]):
        raise ValueError(
            "covariance shape must match the observation dimension"
        )

    return observations, design, covariance


colored_C = make_colored_covariance(colored_rho)
colored_L = linalg.cholesky(colored_C, lower=True)
colored_clean = colored_A @ colored_theta

colored_z = make_rng(
    "q1_colored_noise", replicate=0
).normal(size=colored_s.size)

colored_y = colored_clean + colored_L @ colored_z

### 1.2a Implement the Gaussian log-likelihood

Implement the full log-likelihood for a fixed parameter vector:

$$
\ell(\boldsymbol\theta;\mathbf y)
=-\frac12\left[
N\log(2\pi)+\log|\boldsymbol\Sigma|
+(\mathbf y-\mathbf A\boldsymbol\theta)^T
\boldsymbol\Sigma^{-1}
(\mathbf y-\mathbf A\boldsymbol\theta)
\right].
$$

Use a Cholesky factorization and triangular solves. Do not explicitly invert
the covariance or call a multivariate-normal density function.

Retain the normalization terms. Return one value per observation.

In [ ]:
# TODO-FUNCTION | Task 1.2a | Full Gaussian log-likelihood

def gaussian_linear_loglikelihood(
    observations, design, parameters, covariance
):
    """Evaluate log p(y | parameters).

    observations: shape (N,) or (K, N).
    design: shape (N, P).
    parameters: one fixed vector of shape (P,).
    covariance: shape (N, N).

    Return a scalar for one observation, or shape (K,) for a batch.
    """
    observations, design, covariance = check_linear_inputs(
        observations, design, covariance
    )

    parameters = as_float_array(
        parameters, "parameters", ndim=1
    )

    if parameters.size != design.shape[1]:
        raise ValueError(
            "parameters must match the number of design columns"
        )

    # TODO:
    # 1. Factor the covariance.
    # 2. Compute and whiten the residual(s).
    # 3. Compute the log-determinant from the factor.
    # 4. Return the full log-likelihood.
    raise NotImplementedError("Complete Task 1.2a")

In [ ]:
# PROVIDED | Task 1.2a | Independent likelihood reference checks

likelihood_probe_A = np.array([
    [1.0, 0.0],
    [1.0, 1.0],
    [1.0, 2.0],
    [1.0, 3.0],
])

likelihood_probe_theta = np.array([0.4, -0.2])

likelihood_probe_C = 0.7**np.abs(
    np.arange(4)[:, None] - np.arange(4)[None, :]
)

likelihood_probe_Y = np.array([
    [0.3, 0.1, -0.2, 0.4],
    [1.0, 0.7, 0.1, -0.3],
])

likelihood_expected = stats.multivariate_normal.logpdf(
    likelihood_probe_Y,
    mean=likelihood_probe_A @ likelihood_probe_theta,
    cov=likelihood_probe_C,
)

np.testing.assert_allclose(
    gaussian_linear_loglikelihood(
        likelihood_probe_Y,
        likelihood_probe_A,
        likelihood_probe_theta,
        likelihood_probe_C,
    ),
    likelihood_expected,
    atol=1e-10,
)

assert np.ndim(
    gaussian_linear_loglikelihood(
        likelihood_probe_Y[0],
        likelihood_probe_A,
        likelihood_probe_theta,
        likelihood_probe_C,
    )
) == 0

print("Gaussian log-likelihood checks passed.")

### 1.2b Estimate by whitening

For $\boldsymbol\Sigma=\mathbf L\mathbf L^T$, define

$$
\widetilde{\mathbf y}=\mathbf L^{-1}\mathbf y,
\qquad
\widetilde{\mathbf A}=\mathbf L^{-1}\mathbf A.
$$

Solve least squares in the whitened coordinates and return the information matrix:

$$
\mathcal I
=\widetilde{\mathbf A}^{T}\widetilde{\mathbf A}.
$$

Use triangular solves and `np.linalg.lstsq`. Do not explicitly invert matrices.
Reject rank-deficient designs: the parameter MLE is then non-unique.

In [ ]:
# TODO-FUNCTION | Task 1.2b | Whitened Gaussian MLE

def gaussian_linear_mle(observations, design, covariance):
    """Estimate parameters and return the model's information matrix.

    observations: shape (N,) or (K, N).
    design: shape (N, P), requiring full column rank.
    covariance: shape (N, N).

    Return:
    estimates:
        Shape (P,) for one observation, or (K, P) for a batch.
    information:
        Shape (P, P). This is Fisher information when the supplied
        covariance is the true covariance.
    """
    observations, design, covariance = check_linear_inputs(
        observations, design, covariance
    )

    # TODO:
    # 1. Factor the covariance.
    # 2. Whiten the design and observations.
    # 3. Solve least squares and check the returned rank.
    # 4. Compute the information matrix.
    # 5. Return estimates with the required orientation, and information.
    raise NotImplementedError("Complete Task 1.2b")

In [ ]:
# PROVIDED | Task 1.2b | White-noise limit, scaling, geometry, and rank

linear_probe_fit, linear_probe_info = gaussian_linear_mle(
    likelihood_probe_Y,
    likelihood_probe_A,
    np.eye(4),
)

np.testing.assert_allclose(
    linear_probe_fit,
    np.linalg.lstsq(
        likelihood_probe_A,
        likelihood_probe_Y.T,
        rcond=None,
    )[0].T,
    atol=1e-12,
)

np.testing.assert_allclose(
    linear_probe_info,
    likelihood_probe_A.T @ likelihood_probe_A,
    atol=1e-12,
)

colored_fit, colored_info = gaussian_linear_mle(
    colored_y, colored_A, colored_C
)

colored_scaled_fit, colored_scaled_info = gaussian_linear_mle(
    colored_y, colored_A, 4.0 * colored_C
)

np.testing.assert_allclose(
    colored_scaled_fit, colored_fit, atol=1e-10
)
np.testing.assert_allclose(
    colored_scaled_info, colored_info / 4.0, atol=1e-10
)

colored_white_residual = linalg.solve_triangular(
    colored_L,
    colored_y - colored_A @ colored_fit,
    lower=True,
)

colored_white_design = linalg.solve_triangular(
    colored_L, colored_A, lower=True
)

np.testing.assert_allclose(
    colored_white_design.T @ colored_white_residual,
    np.zeros(2),
    atol=1e-9,
)

try:
    gaussian_linear_mle(
        np.ones(4),
        np.ones((4, 2)),
        np.eye(4),
    )
except ValueError:
    pass
else:
    raise AssertionError(
        "A non-unique MLE must not be reported as unique"
    )

print("Whitened MLE checks passed.")

In [ ]:
# EXPERIMENT | Task 1.2b | Single observation and repeated performance

colored_ols_fit, _ = gaussian_linear_mle(
    colored_y,
    colored_A,
    np.eye(colored_s.size),
)

print("True [amplitude, baseline]:", colored_theta)
print(
    pd.DataFrame(
        [colored_ols_fit, colored_fit],
        index=["OLS", "GLS / correct MLE"],
        columns=["a_hat", "b_hat"],
    ).to_string(float_format=lambda x: f"{x:.6f}")
)

colored_Z = make_rng(
    "q1_colored_noise", replicate=1
).normal(size=(colored_trials, colored_s.size))

colored_Y = colored_clean[None, :] + colored_Z @ colored_L.T

colored_gls_samples, colored_info = gaussian_linear_mle(
    colored_Y, colored_A, colored_C
)

colored_ols_samples, _ = gaussian_linear_mle(
    colored_Y,
    colored_A,
    np.eye(colored_s.size),
)

colored_naive_samples = estimate_amplitude_mle(
    colored_Y, colored_s
)

colored_reference_cov = linalg.solve(
    colored_info,
    np.eye(2),
    assume_a="pos",
)

colored_ols_gain = np.linalg.lstsq(
    colored_A,
    np.eye(colored_s.size),
    rcond=None,
)[0]

colored_ols_cov = (
    colored_ols_gain @ colored_C @ colored_ols_gain.T
)

colored_rows = []

for name, values in [
    ("Ignore baseline", colored_naive_samples),
    ("OLS", colored_ols_samples[:, 0]),
    ("GLS / correct MLE", colored_gls_samples[:, 0]),
]:
    colored_rows.append({
        "method": name,
        **summarize_estimates(values, colored_theta[0]),
    })

print("\nAmplitude performance:")
print(
    pd.DataFrame(colored_rows).to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\nGLS empirical covariance:")
print(np.cov(colored_gls_samples, rowvar=False, ddof=1))

print("\nInverse Fisher information:")
print(colored_reference_cov)

assert (
    gaussian_linear_loglikelihood(
        colored_y, colored_A, colored_fit, colored_C
    )
    >= gaussian_linear_loglikelihood(
        colored_y, colored_A, colored_ols_fit, colored_C
    ) - 1e-9
)

In [ ]:
# PLOT/EVALUATION | Figure 1.2A | Fitted waveform and joint uncertainty

def sampling_ellipse(covariance, probability=0.95):
    """Return a Gaussian error-distribution contour centered at zero."""
    eigenvalues, eigenvectors = linalg.eigh(covariance)

    angles = np.linspace(0, 2 * np.pi, 250)
    circle = np.vstack([
        np.cos(angles),
        np.sin(angles),
    ])

    return (
        np.sqrt(stats.chi2.ppf(probability, df=2))
        * (
            eigenvectors
            @ np.diag(np.sqrt(eigenvalues))
            @ circle
        )
    )


fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.0))

axes[0].plot(
    warmup_t,
    colored_y,
    color=COLORS["data"],
    lw=1,
    label="Observation",
)
axes[0].plot(
    warmup_t,
    colored_clean,
    color=COLORS["truth"],
    lw=2,
    label="True mean waveform",
)
axes[0].plot(
    warmup_t,
    colored_A @ colored_ols_fit,
    color=COLORS["peak"],
    ls="--",
    label="OLS fit",
)
axes[0].plot(
    warmup_t,
    colored_A @ colored_fit,
    color=COLORS["mle"],
    label="GLS fit",
)
axes[0].axhline(
    colored_theta[1],
    color=COLORS["truth"],
    ls=":",
    label="True baseline",
)
axes[0].set(
    xlabel="Sample coordinate",
    ylabel="Signal value",
    title="Estimate the amplitude and baseline",
)

for name, estimates, covariance, color in [
    (
        "OLS",
        colored_ols_samples,
        colored_ols_cov,
        COLORS["peak"],
    ),
    (
        "GLS",
        colored_gls_samples,
        colored_reference_cov,
        COLORS["mle"],
    ),
]:
    errors = estimates[:400] - colored_theta

    axes[1].scatter(
        errors[:, 0],
        errors[:, 1],
        s=8,
        alpha=0.15,
        color=color,
    )

    ellipse = sampling_ellipse(covariance)

    axes[1].plot(
        ellipse[0],
        ellipse[1],
        color=color,
        lw=2,
        label=f"{name}: 95% sampling contour",
    )

axes[1].axhline(0, color="gray", lw=0.6)
axes[1].axvline(0, color="gray", lw=0.6)
axes[1].set(
    xlabel="Amplitude error",
    ylabel="Baseline error",
    title="Joint uncertainty across repeated measurements",
)

for ax in axes:
    ax.legend(fontsize=7.5)

finish_figure(fig)

### 1.2c Investigate noise geometry

Repeat the comparison across the supplied correlation values.
Keep the signal, parameters, trial count, and marginal noise variance fixed.

Examine amplitude and baseline uncertainty separately. Determine whether
increasing correlation affects both parameters in the same way.

In [ ]:
# EXPERIMENT | Task 1.2c | Correlation sweep at fixed marginal variance

colored_rho_grid = np.array([0.0, 0.5, 0.85, 0.95, 0.99])
colored_sweep_rows = []

for replicate, rho in enumerate(colored_rho_grid, start=10):
    covariance = make_colored_covariance(float(rho))
    factor = linalg.cholesky(covariance, lower=True)

    Z = make_rng(
        "q1_colored_noise",
        replicate=replicate,
    ).normal(size=(colored_trials, colored_s.size))

    observations = colored_clean[None, :] + Z @ factor.T

    gls, information = gaussian_linear_mle(
        observations, colored_A, covariance
    )

    ols, _ = gaussian_linear_mle(
        observations,
        colored_A,
        np.eye(colored_s.size),
    )

    gls_reference = linalg.solve(
        information,
        np.eye(2),
        assume_a="pos",
    )

    ols_reference = (
        colored_ols_gain @ covariance @ colored_ols_gain.T
    )

    row = {"rho": float(rho)}

    for j, parameter in enumerate(("a", "b")):
        row[f"gls_{parameter}_var"] = float(
            gls[:, j].var(ddof=1)
        )
        row[f"gls_{parameter}_theory"] = float(
            gls_reference[j, j]
        )
        row[f"ols_{parameter}_var"] = float(
            ols[:, j].var(ddof=1)
        )
        row[f"ols_{parameter}_theory"] = float(
            ols_reference[j, j]
        )

    colored_sweep_rows.append(row)

colored_sweep = pd.DataFrame(colored_sweep_rows)

print("Amplitude variance versus correlation:")
print(
    colored_sweep[
        [
            "rho",
            "gls_a_var",
            "gls_a_theory",
            "ols_a_var",
            "ols_a_theory",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\nBaseline variance versus correlation:")
print(
    colored_sweep[
        [
            "rho",
            "gls_b_var",
            "gls_b_theory",
            "ols_b_var",
            "ols_b_theory",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

In [ ]:
# PLOT/EVALUATION | Figure 1.2B | Parameter-specific uncertainty

fig, axes = plt.subplots(1, 2, figsize=(11.8, 3.8))

for ax, parameter, title in zip(
    axes,
    ("a", "b"),
    ("Amplitude uncertainty", "Baseline uncertainty"),
):
    for method, color in [
        ("ols", COLORS["peak"]),
        ("gls", COLORS["mle"]),
    ]:
        ax.plot(
            colored_sweep["rho"],
            colored_sweep[f"{method}_{parameter}_theory"],
            color=color,
            lw=1.6,
            label=f"{method.upper()} theory",
        )

        ax.plot(
            colored_sweep["rho"],
            colored_sweep[f"{method}_{parameter}_var"],
            "o",
            color=color,
            label=f"{method.upper()} empirical",
        )

    ax.set(
        xlabel="Noise correlation parameter ρ",
        ylabel="Estimator variance",
        title=title,
    )
    ax.legend(fontsize=8)

finish_figure(fig)

## 1.3 Gaussian Bayesian estimation and prior mismatch

Retain the linear observation model and introduce a Gaussian parameter prior:

$$
\boldsymbol\theta\sim\mathcal N(\boldsymbol\mu_0,\mathbf P_0).
$$

Implement the posterior, investigate prior strength and mismatch, and distinguish
performance at a fixed parameter from performance averaged over the prior.

For a Gaussian posterior, MAP and MMSE coincide numerically.

In [ ]:
# PROVIDED | Task 1.3 | Observation model and prior configurations

bayes_A = colored_A.copy()
bayes_C = colored_C.copy()

# The population distribution used in the prior-averaged experiment.
bayes_prior_mean = np.array([1.3, 0.2])
bayes_prior_sd = np.array([0.4, 0.2])

bayes_prior_cov = (
    np.outer(bayes_prior_sd, bayes_prior_sd)
    * np.array([
        [1.0, 0.3],
        [0.3, 1.0],
    ])
)

# A deliberately incorrect assumed prior mean.
bayes_shifted_mean = (
    bayes_prior_mean + np.array([-1.0, 0.4])
)

# Multipliers of prior STANDARD DEVIATIONS.
# Therefore the covariance multiplier is scale**2.
bayes_scale_grid = np.array([
    0.25, 0.5, 1.0, 2.0, 4.0, 16.0
])

bayes_trials = Q1_CONFIG["n_trials"]

### 1.3a Implement the Gaussian posterior

Use the posterior precision, covariance, and mean:

$$
\mathbf J_{\mathrm{post}}
=\mathbf P_0^{-1}+\mathbf A^T\boldsymbol\Sigma^{-1}\mathbf A,
\qquad
\mathbf P_{\mathrm{post}}=\mathbf J_{\mathrm{post}}^{-1},
$$

$$
\boldsymbol\mu_{\mathrm{post}}
=\mathbf P_{\mathrm{post}}
\left(\mathbf P_0^{-1}\boldsymbol\mu_0
+\mathbf A^T\boldsymbol\Sigma^{-1}\mathbf y\right).
$$

Use Cholesky-based solves rather than `np.linalg.inv`.
A proper prior permits inference even when the design is rank deficient.

In [ ]:
# TODO-FUNCTION | Task 1.3a | Gaussian posterior

def gaussian_linear_posterior(
    observations,
    design,
    covariance,
    prior_mean,
    prior_covariance,
):
    """Return the posterior mean and covariance.

    observations: shape (N,) or (K, N).
    design: shape (N, P); full column rank is not required.
    covariance: positive definite, shape (N, N).
    prior_mean: shape (P,).
    prior_covariance: positive definite, shape (P, P).

    Returns
    -------
    posterior_mean:
        Shape (P,) for one observation, or (K, P) for a batch.
    posterior_covariance:
        Shape (P, P), shared by all observations in the batch.
    """
    observations, design, covariance = check_linear_inputs(
        observations, design, covariance
    )

    prior_mean = as_float_array(
        prior_mean, "prior_mean", ndim=1
    )
    prior_covariance = validate_spd(
        prior_covariance, "prior_covariance"
    )

    p = design.shape[1]

    if (
        prior_mean.shape != (p,)
        or prior_covariance.shape != (p, p)
    ):
        raise ValueError(
            "Prior dimensions must match the design columns"
        )

    # TODO:
    # 1. Whiten the design and observations.
    # 2. Compute the prior and posterior precision matrices using solves.
    # 3. Solve for posterior mean(s) and covariance.
    # 4. Return the required shapes.
    raise NotImplementedError("Complete Task 1.3a")

In [ ]:
# PROVIDED | Task 1.3a | Analytic posterior checks

posterior_probe_mean = np.array([0.5, -0.2])
posterior_probe_prior_var = np.array([0.8, 0.3])
posterior_probe_noise_var = np.array([0.2, 0.6])

posterior_probe_Y = np.array([
    [1.0, -1.0],
    [0.2, 0.4],
])

posterior_probe_fit, posterior_probe_cov = (
    gaussian_linear_posterior(
        posterior_probe_Y,
        np.eye(2),
        np.diag(posterior_probe_noise_var),
        posterior_probe_mean,
        np.diag(posterior_probe_prior_var),
    )
)

# Independent scalar formulas for this diagonal identity model.
posterior_probe_weight = (
    posterior_probe_prior_var
    / (posterior_probe_prior_var + posterior_probe_noise_var)
)

np.testing.assert_allclose(
    posterior_probe_fit,
    posterior_probe_mean
    + (posterior_probe_Y - posterior_probe_mean)
    * posterior_probe_weight,
    atol=1e-12,
)

np.testing.assert_allclose(
    posterior_probe_cov,
    np.diag(
        posterior_probe_prior_var
        * posterior_probe_noise_var
        / (
            posterior_probe_prior_var
            + posterior_probe_noise_var
        )
    ),
    atol=1e-12,
)

posterior_probe_single, _ = gaussian_linear_posterior(
    posterior_probe_Y[0],
    np.eye(2),
    np.diag(posterior_probe_noise_var),
    posterior_probe_mean,
    np.diag(posterior_probe_prior_var),
)

assert posterior_probe_single.shape == (2,)

np.testing.assert_allclose(
    posterior_probe_single,
    posterior_probe_fit[0],
)

# A zero design carries no parameter information.
no_data_mean, no_data_cov = gaussian_linear_posterior(
    posterior_probe_Y[0],
    np.zeros((2, 2)),
    np.eye(2),
    posterior_probe_mean,
    np.diag(posterior_probe_prior_var),
)

np.testing.assert_allclose(
    no_data_mean, posterior_probe_mean, atol=1e-12
)
np.testing.assert_allclose(
    no_data_cov,
    np.diag(posterior_probe_prior_var),
    atol=1e-12,
)

print("Gaussian posterior checks passed.")

In [ ]:
# EXPERIMENT | Task 1.3a | Scalar warm-up and vector observation

# Return briefly to the scalar model from Task 1.1.
scalar_post_mean, scalar_post_cov = gaussian_linear_posterior(
    warmup_y,
    warmup_s[:, None],
    warmup_sigma**2 * np.eye(warmup_s.size),
    bayes_prior_mean[:1],
    bayes_prior_cov[:1, :1],
)

# Then apply the same function to the amplitude/baseline model.
vector_post_mean, vector_post_cov = gaussian_linear_posterior(
    colored_y,
    bayes_A,
    bayes_C,
    bayes_prior_mean,
    bayes_prior_cov,
)

print("Scalar warm-up:")
print(
    f"MLE = {warmup_mle:.6f}; "
    f"Gaussian MAP = MMSE = {scalar_post_mean[0]:.6f}"
)
print(
    f"Posterior SD = "
    f"{np.sqrt(scalar_post_cov[0, 0]):.6f}"
)

print("\nVector observation:")
print(
    pd.DataFrame(
        [colored_fit, vector_post_mean],
        index=["MLE", "Gaussian MAP / MMSE"],
        columns=["a_hat", "b_hat"],
    ).to_string(float_format=lambda x: f"{x:.6f}")
)

### 1.3b Compare conditional and prior-averaged performance

Use two experiments:

- **Fixed parameter:** reuse Task 1.2 observations with the same true parameter.
- **Prior-averaged:** independently draw a new parameter and measurement noise
  for every trial, then generate its observation.

Implement the second simulation. The true population prior remains fixed
while the inference method's assumed prior mean and covariance are varied.

Total parameter MSE means the average squared Euclidean parameter error.

In [ ]:
# TODO-FUNCTION | Task 1.3b | Prior-predictive simulation

def simulate_prior_predictive(
    design,
    covariance,
    prior_mean,
    prior_covariance,
    trials,
    rng,
):
    """Draw parameters and observations under the specified model.

    Return
    ------
    parameters: shape (trials, P).
    observations: shape (trials, N).

    Draw a new parameter vector and independent measurement noise
    for every trial. Use the supplied generator without reseeding.
    """
    design = as_float_array(design, "design", ndim=2)
    covariance = validate_spd(covariance)

    prior_mean = as_float_array(
        prior_mean, "prior_mean", ndim=1
    )
    prior_covariance = validate_spd(
        prior_covariance, "prior_covariance"
    )

    n, p = design.shape

    if (
        covariance.shape != (n, n)
        or prior_mean.shape != (p,)
        or prior_covariance.shape != (p, p)
    ):
        raise ValueError("Model dimensions do not agree")

    if not isinstance(trials, (int, np.integer)) or trials < 2:
        raise ValueError("trials must be an integer >= 2")

    if not isinstance(rng, np.random.Generator):
        raise TypeError("rng must be a NumPy Generator")

    # TODO:
    # 1. Draw parameter vectors with the specified mean and covariance.
    # 2. Independently draw measurement noise.
    # 3. Generate observations and return both arrays.
    raise NotImplementedError("Complete Task 1.3b")

In [ ]:
# PROVIDED | Task 1.3b | Simulation interface and reproducibility checks

prior_check_theta, prior_check_Y = simulate_prior_predictive(
    bayes_A,
    bayes_C,
    bayes_prior_mean,
    bayes_prior_cov,
    30,
    make_rng("q1_prior", replicate=999),
)

prior_check_theta2, prior_check_Y2 = simulate_prior_predictive(
    bayes_A,
    bayes_C,
    bayes_prior_mean,
    bayes_prior_cov,
    30,
    make_rng("q1_prior", replicate=999),
)

assert prior_check_theta.shape == (30, 2)
assert prior_check_Y.shape == (30, bayes_A.shape[0])

assert np.isfinite(prior_check_theta).all()
assert np.isfinite(prior_check_Y).all()
assert np.all(prior_check_theta.std(axis=0) > 0)

np.testing.assert_array_equal(
    prior_check_theta, prior_check_theta2
)
np.testing.assert_array_equal(
    prior_check_Y, prior_check_Y2
)

print("Prior-predictive simulation checks passed.")

In [ ]:
# EXPERIMENT | Task 1.3b | Prior strength, risk, and coverage

def gaussian_performance(
    estimates, truth, posterior_covariance
):
    """Summarize errors and coverage of Gaussian 95% joint regions.

    This supplied evaluator expects one parameter vector per row.
    Coverage is interpreted below across prior-predictive trials.
    """
    errors = np.asarray(estimates) - np.asarray(truth)

    L = linalg.cholesky(
        posterior_covariance, lower=True
    )

    standardized = linalg.solve_triangular(
        L, errors.T, lower=True
    ).T

    inside = (
        np.sum(standardized**2, axis=1)
        <= stats.chi2.ppf(0.95, df=errors.shape[1])
    )

    return {
        "total_mse": float(
            np.mean(np.sum(errors**2, axis=1))
        ),
        "coverage": float(inside.mean()),
    }


# Generate the population ONCE, using the true population prior.
bayes_population_theta, bayes_population_Y = (
    simulate_prior_predictive(
        bayes_A,
        bayes_C,
        bayes_prior_mean,
        bayes_prior_cov,
        bayes_trials,
        make_rng("q1_prior", replicate=0),
    )
)

bayes_population_mle, _ = gaussian_linear_mle(
    bayes_population_Y, bayes_A, bayes_C
)

bayes_fixed_mle_risk = float(
    np.mean(
        np.sum(
            (colored_gls_samples - colored_theta)**2,
            axis=1,
        )
    )
)

bayes_population_mle_risk = float(
    np.mean(
        np.sum(
            (bayes_population_mle - bayes_population_theta)**2,
            axis=1,
        )
    )
)

bayes_rows = []

for name, mean in [
    ("Correct mean", bayes_prior_mean),
    ("Shifted mean", bayes_shifted_mean),
]:
    for scale in bayes_scale_grid:
        assumed_cov = scale**2 * bayes_prior_cov

        fixed_mean, post_cov = gaussian_linear_posterior(
            colored_Y,
            bayes_A,
            bayes_C,
            mean,
            assumed_cov,
        )

        population_mean, _ = gaussian_linear_posterior(
            bayes_population_Y,
            bayes_A,
            bayes_C,
            mean,
            assumed_cov,
        )

        performance = gaussian_performance(
            population_mean,
            bayes_population_theta,
            post_cov,
        )

        bayes_rows.append({
            "prior_mean": name,
            "sd_scale": float(scale),
            "fixed_mse": float(
                np.mean(
                    np.sum(
                        (fixed_mean - colored_theta)**2,
                        axis=1,
                    )
                )
            ),
            "population_mse": performance["total_mse"],
            "joint_coverage": performance["coverage"],
            "reported_uncertainty": float(
                np.trace(post_cov)
            ),
        })

bayes_results = pd.DataFrame(bayes_rows)

bayes_matched = bayes_results[
    (bayes_results["prior_mean"] == "Correct mean")
    & (bayes_results["sd_scale"] == 1.0)
].iloc[0]

print(
    "Fixed-parameter MLE total MSE:",
    f"{bayes_fixed_mle_risk:.6f}",
)
print(
    "Prior-averaged MLE total MSE:",
    f"{bayes_population_mle_risk:.6f}",
)

print("\nRisk and coverage:")
print(
    bayes_results.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# Under the matched model, Bayes risk equals trace(P_post).
print(
    "\nMatched-prior empirical Bayes risk:",
    f"{bayes_matched['population_mse']:.6f}",
)
print(
    "Matched-prior theoretical Bayes risk:",
    f"{bayes_matched['reported_uncertainty']:.6f}",
)

In [ ]:
# PLOT/EVALUATION | Figure 1.3A | Two different risk comparisons

fig, axes = plt.subplots(1, 2, figsize=(11.8, 3.8))

for name, color in [
    ("Correct mean", COLORS["map"]),
    ("Shifted mean", COLORS["peak"]),
]:
    rows = bayes_results[
        bayes_results["prior_mean"] == name
    ]

    axes[0].semilogx(
        rows["sd_scale"],
        rows["fixed_mse"],
        "o-",
        color=color,
        label=name,
    )

    axes[1].semilogx(
        rows["sd_scale"],
        rows["population_mse"],
        "o-",
        color=color,
        label=name,
    )

for ax, risk, title in zip(
    axes,
    [bayes_fixed_mle_risk, bayes_population_mle_risk],
    [
        "Fixed true parameter",
        "Parameters drawn from the true prior",
    ],
):
    ax.axhline(
        risk,
        color=COLORS["mle"],
        ls="--",
        label="MLE",
    )

    ax.axvline(
        1.0,
        color="gray",
        ls=":",
        lw=1,
        label="True population covariance scale",
    )

    ax.set(
        xlabel="Assumed prior SD multiplier",
        ylabel="Total parameter MSE",
        title=title,
    )
    ax.legend(fontsize=7.5)

finish_figure(fig)

### 1.3c Inspect uncertainty and non-identifiability

First, compare nominal 95% joint credible-region coverage across the
prior-predictive trials.

Then change the observation model to two identical templates:

$$
\mathbf y=(\theta_1+\theta_2)\mathbf s+\mathbf n.
$$

These parameters are now two component amplitudes, not amplitude and baseline.
Use an isotropic Gaussian prior and compare prior/posterior uncertainty along
the normalized sum and difference directions.

In [ ]:
# EXPERIMENT | Task 1.3c | A proper posterior with a rank-deficient design

null_A = np.column_stack([colored_s, colored_s])

null_prior_mean = np.zeros(2)
null_prior_cov = np.eye(2)

null_truth = np.array([1.0, 0.7])

null_noise = make_rng(
    "q1_prior", replicate=20
).normal(size=colored_s.size)

null_y = (
    null_A @ null_truth
    + colored_L @ null_noise
)

null_post_mean, null_post_cov = gaussian_linear_posterior(
    null_y,
    null_A,
    bayes_C,
    null_prior_mean,
    null_prior_cov,
)

# Columns are orthonormal sum and difference directions.
null_directions = np.column_stack([
    np.array([1.0, 1.0]) / np.sqrt(2.0),
    np.array([1.0, -1.0]) / np.sqrt(2.0),
])

null_prior_var = np.diag(
    null_directions.T
    @ null_prior_cov
    @ null_directions
)

null_post_var = np.diag(
    null_directions.T
    @ null_post_cov
    @ null_directions
)

# Under this isotropic prior, the difference receives no information.
np.testing.assert_allclose(
    null_post_var[1],
    null_prior_var[1],
    atol=1e-10,
)
assert null_post_var[0] < null_prior_var[0]

print("Identical-template posterior mean:", null_post_mean)

print(
    pd.DataFrame({
        "direction": [
            "Sum / sqrt(2)",
            "Difference / sqrt(2)",
        ],
        "prior_var": null_prior_var,
        "posterior_var": null_post_var,
    }).to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

In [ ]:
# PLOT/EVALUATION | Figure 1.3B | Calibration and information directions

coverage_cases = [
    ("Matched prior", "Correct mean", 1.0),
    ("Shifted mean", "Shifted mean", 1.0),
    ("Shifted + narrow", "Shifted mean", 0.25),
]

coverage_values = []

for label, name, scale in coverage_cases:
    selected = bayes_results.loc[
        (bayes_results["prior_mean"] == name)
        & (bayes_results["sd_scale"] == scale),
        "joint_coverage",
    ]

    coverage_values.append(float(selected.iloc[0]))

fig, axes = plt.subplots(1, 2, figsize=(11.8, 3.8))

axes[0].bar(
    np.arange(3),
    coverage_values,
    color=[
        COLORS["map"],
        COLORS["peak"],
        COLORS["data"],
    ],
)

axes[0].axhline(
    0.95,
    color=COLORS["truth"],
    ls="--",
    label="Nominal 95%",
)

axes[0].set_xticks(
    np.arange(3),
    [case[0] for case in coverage_cases],
)

for position, value in enumerate(coverage_values):
    axes[0].text(
        position,
        value + 0.015,
        f"{100 * value:.1f}%",
        ha="center",
        fontsize=8,
    )

axes[0].set(
    ylim=(0, 1.06),
    ylabel="Empirical joint coverage",
    title="Coverage across prior-predictive trials",
)
axes[0].legend(fontsize=8)

positions = np.arange(2)
width = 0.34

axes[1].bar(
    positions - width / 2,
    null_prior_var,
    width,
    color=COLORS["data"],
    label="Prior",
)
axes[1].bar(
    positions + width / 2,
    null_post_var,
    width,
    color=COLORS["map"],
    label="Posterior",
)

axes[1].set_xticks(
    positions,
    ["Sum direction", "Difference direction"],
)
axes[1].set(
    ylabel="Projected parameter variance",
    title="Two identical templates",
)
axes[1].legend(fontsize=8)

finish_figure(fig)

## 1.4 One posterior, different losses and decisions

The baseline is now known and removed. The amplitude has two possible states:

$$
a\in\{0,A\},\qquad P(a=A)=\pi,
\qquad
\mathbf y=a\mathbf s+\mathbf n.
$$

The waveform, active amplitude, prior probability, and noise covariance are known.

Compute the discrete posterior, then compare MAP classification, MMSE amplitude
estimation, and classification with unequal error costs.

In [ ]:
# PROVIDED | Task 1.4 | Two-state amplitude model

binary_s = colored_s.copy()
binary_C = colored_C.copy()

binary_active_amplitude = 1.0
binary_prior_active = 0.20

binary_false_alarm_cost = 1.0
binary_miss_cost = 10.0

binary_trials = 2 * Q1_CONFIG["n_trials"]
binary_rng = make_rng("q1_discrete", replicate=0)

binary_labels = binary_rng.binomial(
    n=1,
    p=binary_prior_active,
    size=binary_trials,
)

binary_true_amplitudes = (
    binary_active_amplitude * binary_labels
)

binary_L = linalg.cholesky(
    binary_C, lower=True
)

binary_noise = (
    binary_rng.normal(
        size=(binary_trials, binary_s.size)
    )
    @ binary_L.T
)

binary_Y = (
    binary_true_amplitudes[:, None]
    * binary_s[None, :]
    + binary_noise
)

### 1.4a Compute posterior state probabilities


For each observation, compute the probabilities of the inactive and active states:

$$
P(a=a_k\mid\mathbf y)
\propto p(\mathbf y\mid a_k)P(a=a_k),
\qquad a_0=0,\quad a_1=A.
$$

Reuse `gaussian_linear_loglikelihood`. Normalize in the log domain using
`special.logsumexp` or an equivalent stable calculation.

Return probabilities in the order `[inactive, active]`.

In [ ]:
# TODO-FUNCTION | Task 1.4a | Discrete amplitude posterior

def binary_amplitude_posterior(
    observations,
    template,
    covariance,
    active_amplitude,
    prior_active,
):
    """Return probabilities in the order [inactive, active].

    observations: shape (N,) or (K, N).

    Returns
    -------
    Shape (2,) for one observation, or (K, 2) for a batch.
    """
    template = as_float_array(
        template, "template", ndim=1
    )

    if not np.any(template):
        raise ValueError("template must be nonzero")

    observations, design, covariance = check_linear_inputs(
        observations,
        template[:, None],
        covariance,
    )

    active_amplitude = float(
        as_float_array(
            active_amplitude,
            "active_amplitude",
            ndim=0,
        )
    )
    prior_active = float(
        as_float_array(
            prior_active,
            "prior_active",
            ndim=0,
        )
    )

    if active_amplitude <= 0 or not 0 < prior_active < 1:
        raise ValueError(
            "Require active_amplitude > 0 and 0 < prior_active < 1"
        )

    # TODO:
    # 1. Evaluate the two Gaussian log-likelihoods.
    # 2. Add the corresponding log prior probabilities.
    # 3. Normalize stably and return posterior probabilities.
    raise NotImplementedError("Complete Task 1.4a")

In [ ]:
# PROVIDED | Task 1.4a | Posterior normalization and stability checks

# At the midpoint observation, the two likelihoods are equal.
# Consequently, posterior probabilities equal prior probabilities.
binary_midpoint_posterior = binary_amplitude_posterior(
    0.5 * binary_active_amplitude * binary_s,
    binary_s,
    binary_C,
    binary_active_amplitude,
    binary_prior_active,
)

np.testing.assert_allclose(
    binary_midpoint_posterior,
    [1.0 - binary_prior_active, binary_prior_active],
    atol=1e-10,
)

# Extreme observations expose unstable direct-density calculations.
binary_stress_Y = (
    np.array([-100.0, 0.0, 100.0])[:, None]
    * binary_s[None, :]
)

binary_stress_p = binary_amplitude_posterior(
    binary_stress_Y,
    binary_s,
    binary_C,
    binary_active_amplitude,
    binary_prior_active,
)

assert binary_stress_p.shape == (3, 2)
assert np.isfinite(binary_stress_p).all()

np.testing.assert_allclose(
    binary_stress_p.sum(axis=-1),
    1.0,
    atol=1e-10,
)

assert binary_stress_p[0, 1] < 1e-10
assert binary_stress_p[-1, 1] > 1.0 - 1e-10

print("Binary posterior checks passed.")

### 1.4b Choose an action for each loss

Let $p=P(a=A\mid\mathbf y)$.

- **MAP state:** minimize classification error with equal error costs.
- **MMSE amplitude:** return the posterior mean, allowing intermediate estimates.
- **Minimum-cost state:** compare the posterior decision risks

$$
R(0\mid\mathbf y)=C_{\mathrm{miss}}p,
\qquad
R(1\mid\mathbf y)=C_{\mathrm{FA}}(1-p).
$$

Return state labels as 0 or 1, and amplitude estimates in amplitude units.
Resolve exact decision ties in favor of state 0.

In [ ]:
# TODO-FUNCTION | Task 1.4b | Loss-dependent Bayes actions

def binary_amplitude_actions(
    probabilities,
    active_amplitude,
    false_alarm_cost,
    miss_cost,
):
    """Return MAP states, MMSE amplitudes, and minimum-cost states.

    probabilities: shape (2,) or (K, 2), ordered [inactive, active].

    Return a dictionary with keys:
        "map_state"
        "mmse_amplitude"
        "cost_state"

    Outputs are scalars for one posterior, or shape (K,) for a batch.
    """
    probabilities = as_float_array(
        probabilities, "probabilities"
    )

    if (
        probabilities.ndim not in (1, 2)
        or probabilities.shape[-1] != 2
    ):
        raise ValueError(
            "probabilities must have shape (2,) or (K, 2)"
        )

    if (
        np.any((probabilities < 0) | (probabilities > 1))
        or not np.allclose(
            probabilities.sum(axis=-1),
            1.0,
            rtol=1e-10,
            atol=1e-12,
        )
    ):
        raise ValueError(
            "Posterior probabilities must be nonnegative and sum to one"
        )

    active_amplitude = float(
        as_float_array(
            active_amplitude, "active_amplitude", ndim=0
        )
    )
    false_alarm_cost = float(
        as_float_array(
            false_alarm_cost, "false_alarm_cost", ndim=0
        )
    )
    miss_cost = float(
        as_float_array(miss_cost, "miss_cost", ndim=0)
    )

    if min(active_amplitude, false_alarm_cost, miss_cost) <= 0:
        raise ValueError(
            "Amplitude and both costs must be positive"
        )

    # TODO: compute and return the three required outputs.
    raise NotImplementedError("Complete Task 1.4b")

In [ ]:
# PROVIDED | Task 1.4b | Decision and estimation checks

binary_action_probe = binary_amplitude_actions(
    np.array([
        [0.98, 0.02],
        [0.40, 0.60],
        [0.50, 0.50],
    ]),
    binary_active_amplitude,
    binary_false_alarm_cost,
    binary_miss_cost,
)

np.testing.assert_array_equal(
    binary_action_probe["map_state"],
    [0, 1, 0],
)

np.testing.assert_allclose(
    binary_action_probe["mmse_amplitude"],
    binary_active_amplitude * np.array([0.02, 0.60, 0.50]),
)

np.testing.assert_array_equal(
    binary_action_probe["cost_state"],
    [0, 1, 1],
)

binary_single_probe = binary_amplitude_actions(
    np.array([0.8, 0.2]),
    binary_active_amplitude,
    binary_false_alarm_cost,
    binary_miss_cost,
)

assert np.ndim(binary_single_probe["map_state"]) == 0

print("Bayes-action checks passed.")

In [ ]:
# EXPERIMENT | Task 1.4 | Compare criteria on paired observations

binary_posterior = binary_amplitude_posterior(
    binary_Y,
    binary_s,
    binary_C,
    binary_active_amplitude,
    binary_prior_active,
)

binary_actions = binary_amplitude_actions(
    binary_posterior,
    binary_active_amplitude,
    binary_false_alarm_cost,
    binary_miss_cost,
)

# Equal priors recover likelihood-only selection over the two states.
# This is a computational identity, not a change to the true population.
binary_equal_prior = binary_amplitude_posterior(
    binary_Y,
    binary_s,
    binary_C,
    binary_active_amplitude,
    0.5,
)

binary_ml_state = np.argmax(
    binary_equal_prior, axis=-1
)

binary_point_estimates = {
    "ML state amplitude": (
        binary_active_amplitude * binary_ml_state
    ),
    "MAP state amplitude": (
        binary_active_amplitude * binary_actions["map_state"]
    ),
    "MMSE amplitude": binary_actions["mmse_amplitude"],
}

binary_p1 = binary_posterior[:, 1]
binary_mse_rows = []

for name, estimate in binary_point_estimates.items():
    posterior_risk = (
        (1.0 - binary_p1) * estimate**2
        + binary_p1
        * (binary_active_amplitude - estimate)**2
    )

    binary_mse_rows.append({
        "estimator": name,
        "empirical_mse": float(
            np.mean(
                (estimate - binary_true_amplitudes)**2
            )
        ),
        "posterior_expected_mse": float(
            posterior_risk.mean()
        ),
    })

binary_mse_table = pd.DataFrame(binary_mse_rows)

binary_policy_rows = []

for name, states in [
    ("ML state", binary_ml_state),
    ("MAP state", binary_actions["map_state"]),
    ("Minimum-cost state", binary_actions["cost_state"]),
]:
    summary = binary_summary(
        binary_labels, states, threshold=0.5
    )

    cost = (
        binary_false_alarm_cost * summary["fp"]
        + binary_miss_cost * summary["fn"]
    )

    posterior_error = np.where(
        states == 0,
        binary_p1,
        1.0 - binary_p1,
    )

    posterior_cost = np.where(
        states == 0,
        binary_miss_cost * binary_p1,
        binary_false_alarm_cost * (1.0 - binary_p1),
    )

    binary_policy_rows.append({
        "policy": name,
        "error_rate": 1.0 - summary["accuracy"],
        "mean_cost": cost / binary_trials,
        "posterior_expected_error": float(
            posterior_error.mean()
        ),
        "posterior_expected_cost": float(
            posterior_cost.mean()
        ),
        "p_fa": summary["p_fa"],
        "p_d": summary["p_d"],
    })

binary_policy_table = pd.DataFrame(binary_policy_rows)

print("Amplitude estimates:")
print(
    binary_mse_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\nState decisions:")
print(
    binary_policy_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print(
    "\nCost-sensitive posterior threshold:",
    binary_false_alarm_cost
    / (binary_false_alarm_cost + binary_miss_cost),
)

In [ ]:
# PLOT/EVALUATION | Figure 1.4A | Posterior response and point estimates

# Use the continuous model only to choose a useful plotting range.
_, binary_information = gaussian_linear_mle(
    binary_s,
    binary_s[:, None],
    binary_C,
)

binary_effective_sd = (
    1.0 / np.sqrt(binary_information[0, 0])
)

binary_coordinate = np.linspace(
    -3.0 * binary_effective_sd,
    binary_active_amplitude + 3.0 * binary_effective_sd,
    500,
)

# Representative observations along the signal direction.
# These are plotting inputs, not another Monte Carlo dataset.
binary_grid_Y = (
    binary_coordinate[:, None] * binary_s[None, :]
)

binary_grid_p = binary_amplitude_posterior(
    binary_grid_Y,
    binary_s,
    binary_C,
    binary_active_amplitude,
    binary_prior_active,
)

binary_grid_equal_p = binary_amplitude_posterior(
    binary_grid_Y,
    binary_s,
    binary_C,
    binary_active_amplitude,
    0.5,
)

binary_grid_actions = binary_amplitude_actions(
    binary_grid_p,
    binary_active_amplitude,
    binary_false_alarm_cost,
    binary_miss_cost,
)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 3.8))

axes[0].plot(
    binary_coordinate,
    binary_grid_p[:, 1],
    color=COLORS["map"],
    label="Prior P(active) = 0.2",
)

axes[0].plot(
    binary_coordinate,
    binary_grid_equal_p[:, 1],
    color=COLORS["data"],
    ls="--",
    label="Prior P(active) = 0.5",
)

axes[0].axhline(
    0.5,
    color=COLORS["truth"],
    ls=":",
    label="MAP threshold",
)

axes[0].axhline(
    binary_false_alarm_cost
    / (binary_false_alarm_cost + binary_miss_cost),
    color=COLORS["peak"],
    ls=":",
    label="Minimum-cost threshold",
)

axes[0].set(
    ylabel="Posterior probability of active state",
    title="Evidence, prior, and decision criterion",
    ylim=(-0.03, 1.03),
)

axes[1].plot(
    binary_coordinate,
    binary_active_amplitude
    * binary_grid_actions["map_state"],
    color=COLORS["map"],
    ls="--",
    label="MAP amplitude",
)

axes[1].plot(
    binary_coordinate,
    binary_grid_actions["mmse_amplitude"],
    color=COLORS["mmse"],
    lw=2,
    label="MMSE amplitude",
)

axes[1].set(
    ylabel="Amplitude estimate",
    title="Hard decision versus soft estimate",
)

for ax in axes:
    ax.set_xlabel("Amplitude coordinate x in y = x s")
    ax.legend(fontsize=7.5)

finish_figure(fig)

In [ ]:
# PLOT/EVALUATION | Figure 1.4B | One posterior and squared-error risk

# Select a reproducible example with appreciable posterior uncertainty.
binary_example_index = int(
    np.argmin(np.abs(binary_p1 - 0.35))
)

binary_example_prob = binary_posterior[
    binary_example_index
]
binary_example_p = float(binary_example_prob[1])

binary_example_map = float(
    binary_active_amplitude
    * binary_actions["map_state"][binary_example_index]
)

binary_example_mmse = float(
    binary_actions["mmse_amplitude"][binary_example_index]
)

binary_candidates = np.linspace(
    -0.2 * binary_active_amplitude,
    1.2 * binary_active_amplitude,
    500,
)

binary_example_risk = (
    (1.0 - binary_example_p) * binary_candidates**2
    + binary_example_p
    * (binary_active_amplitude - binary_candidates)**2
)

fig, axes = plt.subplots(1, 2, figsize=(11.8, 3.8))

axes[0].bar(
    [0.0, binary_active_amplitude],
    binary_example_prob,
    width=0.14 * binary_active_amplitude,
    color=COLORS["map"],
    alpha=0.65,
    label="Posterior state probability",
)

axes[0].axvline(
    binary_example_mmse,
    color=COLORS["mmse"],
    lw=2,
    label="MMSE amplitude",
)

axes[0].set(
    xticks=[0.0, binary_active_amplitude],
    xlabel="Amplitude",
    ylabel="Probability mass",
    ylim=(0, 1),
    title=(
        "One ambiguous observation: "
        f"P(active | y) = {binary_example_p:.3f}"
    ),
)

axes[1].plot(
    binary_candidates,
    binary_example_risk,
    color=COLORS["truth"],
    lw=2,
)

for value, color, label in [
    (binary_example_map, COLORS["map"], "MAP"),
    (binary_example_mmse, COLORS["mmse"], "MMSE"),
]:
    risk = (
        (1.0 - binary_example_p) * value**2
        + binary_example_p
        * (binary_active_amplitude - value)**2
    )

    axes[1].scatter(
        value,
        risk,
        color=color,
        s=45,
        zorder=4,
        label=label,
    )

axes[1].set(
    xlabel="Candidate amplitude estimate",
    ylabel="Posterior expected squared error",
    title="Which point estimate minimizes this loss?",
)

for ax in axes:
    ax.legend(fontsize=8)

finish_figure(fig)

## 1. Analysis

Answer briefly using your numerical results and figures.

1. Explain how using the full waveform, omitting the baseline and using
   incorrect covariance weighting affect estimation. Distinguish bias
   from variance and support your explanation with theoretical references.

2. Why can the preferred prior strength differ between the fixed-parameter
   and prior-averaged experiments? How does prior mismatch affect error
   and credible-region coverage?

3. With two identical templates, which parameter combination is identified
   by the observations? Does a unique posterior establish identifiability
   from the measurements?

4. Why do Gaussian MAP and MMSE coincide, while they generally differ
   in the two-state model? Explain how changing the loss changes the
   optimal estimate or decision.

**Your answers:**

1. ...
2. ...
3. ...
4. ...

# 2. Airfoil Problem

Helpful links:

1_ [Explicit Regularization](https://gradml.mit.edu/supervised/explicit_regularization/)

2_ [Data Leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)

3_ [Cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)

Predict the dataset's scaled sound pressure level from frequency and
operating-condition measurements. Later tasks compare regression and
classification approaches.

These are tabulated acoustic measurements, not audio waveforms.

Source: [UCI Airfoil Self-Noise](https://archive.ics.uci.edu/dataset/291/airfoil+self-noise).
Brooks, Pope, and Marcolini (1989).
DOI: [10.24432/C5VW2C](https://doi.org/10.24432/C5VW2C).
License: CC BY 4.0.

## 2.1 Data and Evaluation Protocol


1. Load the original dataset and audit its structure and data quality.
2. Construct condition groups and reproducible grouped train/validation/test
   partitions using the prescribed protocol.
3. Verify partition integrity and compare group overlap with a row-wise split.
4. Visualize frequency versus condition-group ID across the three partitions, and plot acoustic responses from three training groups against frequency.

Report the data-audit results, partition row/group counts, and overlap comparison.
Reserve test-target analysis and model evaluation for Section 2.5.

### 2.1a Load the data and perform the data-quality audit

In [ ]:
#TODO

### 2.1b Construct groups, create the partitions, and audit separation

In [ ]:
# TODO

### 2.1c Visualize partition membership and training-group responses

Create the two required plots using logarithmic frequency axes.
Report the conditions of the three selected training groups.

In [ ]:
# TODO
# Visualize frequency versus condition-group ID for all partitions.
# Plot acoustic responses from three training groups and report their conditions.

## 2.2 Feature representations and preprocessing (training only)

1. Construct the prescribed `raw`, `log`, and `log_poly2` representations.
2. Implement reusable preprocessing fitted only on the appropriate fitting rows.
   Keep the target in its supplied dB units.
3. Inspect training-design rank, conditioning, feature correlations,
   and standardized singular-value profiles.
4. Create and verify three grouped CV folds within the training partition.

Report a compact geometry table and a figure showing correlations and
singular-value profiles. Training-audit preprocessing must not be reused
for CV scoring.

### 2.2a Implement the feature representations and reusable preprocessing


In [ ]:
# TODO

### 2.2b Audit and visualize training feature/design geometry

In [ ]:
# TODO

### 2.2c Construct and verify grouped cross-validation folds

In [ ]:
# TODO

## 2.3 Regression and regularization

1. Implement OLS and ridge using numerical linear algebra.
   Verify objective normalization and unpenalized-intercept behavior.
2. Fit LASSO using the permitted library implementation.
3. Evaluate all prescribed configurations using grouped CV, fitting fresh
   preprocessing inside every fold.
4. Select one penalty per representation/method pair by pooled CV RMSE.
   Refit the shortlisted candidates on training data and compare on validation.
5. Select the regression configuration using the prescribed validation rule.

Include a fold-specific mean-only baseline and report CV RMSE, validation
RMSE/MAE, and nonzero weight counts for shortlisted candidates.

### 2.3a Implement and verify the regression estimators

In [ ]:
# TODO

### 2.3b Implement grouped CV and shortlist the candidate configurations

In [ ]:
# TODO

### 2.3c Compare shortlisted models on validation and record the selection

In [ ]:
# TODO
# Refit the shortlisted configurations using training data only.
# Compare validation metrics and record the selected regression configuration.

### 2.3d Create the regularization and validation-diagnostic figures

For `log_poly2`, show fitting and grouped-CV RMSE against penalty,
separately for ridge and LASSO. Include the relevant baselines and selected penalties.

For the selected configuration, show validation predictions against observations
and validation residuals against frequency.

Use informative labels and distinguish development results from final test performance.

In [ ]:
# TODO
# Plot the regularization paths and selected-model validation diagnostics.

## 2.4 From prediction to detection

Define the high-response event using the training targets and the specification
in the homework guide.

- Implement the shared-covariance Gaussian classifier and L2-regularized
  logistic regression using the `log` representation.
- Select their regularization parameters by grouped training CV with
  fold-specific preprocessing and pooled log loss.
- Compare their validation scores with predictions from the selected
  regression model.
- Choose validation thresholds satisfying the false-alarm constraint,
  select the final detection method, and record all choices before testing.

Implement the estimators yourself. Numerical optimization and metric utilities
are allowed.

### 2.4a Event Definition and Classifier Implementation
Define the event and implement the two classifiers.

> Derive the Gaussian log-posterior odds and the logistic objective gradient here.

In [ ]:
#TODO

### 2.4b Grouped Cross-Validation and Model Fitting

- Perform grouped CV, select regularization, and fit on the full training set.
- Obtain validation scores for all three methods

In [ ]:
# TODO

### 2.4c Decision Thresholds and Final Model Selection

- Select operating thresholds and the final detection method.
- Record the complete policy before accessing test outcomes.

In [ ]:
# TODO

### 2.4d Validation ROC and Precision–Recall Curves

- Plot validation ROC and precision–recall curves.
- Mark the selected operating points and include appropriate reference lines.

In [ ]:
# TODO

## 2.5 Final evaluation and interpretation

Apply the frozen models and decision policies to the test groups.

- Report regression RMSE and MAE alongside the training-mean baseline.
- Report classification ranking metrics, confusion counts, detection rate,
  false-alarm rate, and precision for all three frozen policies.
- Identify the method selected before testing; do not select a new winner.
- Compare validation and test false-alarm rates.
- Illustrate how precision changes with event prevalence when conditional
  error rates are held fixed.

Do not refit models or adjust thresholds after inspecting test results.

### 2.5a Final Test Evaluation

Evaluate the frozen regression models and detection policies on the test set.

In [ ]:
# TODO

### 2.5b False-Alarm Transfer and Prevalence Sensitivity

Plot false-alarm transfer and hypothetical prevalence sensitivity.

In [ ]:
# TODO

## 2. Analysis


Answer briefly using your numerical results and figures.

1. What different generalization tasks do row-wise and grouped splitting
   assess? Why must preprocessing be fitted separately inside each CV fold?

2. Explain the effects of feature representation and regularization.
   Distinguish numerical conditioning, coefficient sparsity and predictive
   performance. Does a small score difference establish a clear advantage?

3. Distinguish the event cutoff from the model decision thresholds.
   Does the comparison of Gaussian classification, logistic regression
   and regression-based detection isolate the effect of the learning method?

4. Did the validation false-alarm constraint transfer to test?
   Distinguish changes in event prevalence from changes in conditional
   error rates. State one evaluation limitation and one justified next
   experiment without tuning against the inspected test results.

**Your answers:**

1. ...
2. ...
3. ...
4. ...

# 3. Recognizing Modulation from Noisy Signal Bursts

Helpful links:

1_ [Mathematical formulation of the LDA and QDA](https://scikit-learn.org/stable/modules/lda_qda.html#mathematical-formulation-of-the-lda-and-qda-classifiers)

2_ [Probability calibration](https://scikit-learn.org/stable/modules/calibration.html?utm_source=chatgpt.com)

A receiver observes short BPSK, QPSK and 16-QAM signal bursts with
unknown gain and phase and additive noise.

Using the supplied generator and feature extractor, implement LDA and QDA,
compare their test performance, and assess the quality of their
predicted probabilities.

In [ ]:
# PROVIDED — run without modification.

Q3_CLASS_NAMES = ("BPSK", "QPSK", "16-QAM")

Q3_FEATURE_NAMES = (
    "Normalized second-moment magnitude",
    "Normalized fourth moment",
)

Q3_CONFIG = {
    "burst_length": 128,
    "class_priors": np.array([0.40, 0.35, 0.25]),
    "development_snr_db": (5.0, 15.0),
    "sizes": {
        "train": 600,
        "test": 600,
    },
}

Q3_STREAMS = {
    "train": 0,
    "test": 1,
}


def q3_rng(partition):
    return np.random.default_rng(
        np.random.SeedSequence(
            [BASE_SEED, 300, Q3_STREAMS[partition]]
        )
    )

In [ ]:
# PROVIDED — run without modification.

def simulate_q3_bursts(n_bursts, snr_db_range, rng):
    if (
        not isinstance(n_bursts, (int, np.integer))
        or n_bursts <= 0
    ):
        raise ValueError("n_bursts must be a positive integer.")

    limits = np.asarray(snr_db_range, dtype=float)

    if (
        limits.shape != (2,)
        or not np.isfinite(limits).all()
        or limits[0] > limits[1]
    ):
        raise ValueError("Expected two finite, ordered SNR limits.")

    length = Q3_CONFIG["burst_length"]

    labels = rng.choice(
        3,
        size=n_bursts,
        p=Q3_CONFIG["class_priors"],
    )

    symbols = np.empty((n_bursts, length), dtype=complex)

    for k in range(3):
        rows = np.flatnonzero(labels == k)
        shape = (len(rows), length)

        if k == 0:
            symbols[rows] = rng.choice(
                [-1.0, 1.0], size=shape
            )

        elif k == 1:
            symbols[rows] = (
                rng.choice([-1.0, 1.0], size=shape)
                + 1j * rng.choice([-1.0, 1.0], size=shape)
            ) / np.sqrt(2.0)

        else:
            levels = [-3.0, -1.0, 1.0, 3.0]

            symbols[rows] = (
                rng.choice(levels, size=shape)
                + 1j * rng.choice(levels, size=shape)
            ) / np.sqrt(10.0)

    gain = np.exp(
        rng.uniform(
            np.log(0.5), np.log(2.0), size=n_bursts
        )
    )
    phase = rng.uniform(-np.pi, np.pi, size=n_bursts)
    snr_db = rng.uniform(limits[0], limits[1], size=n_bursts)

    noise_scale = (
        gain * 10.0 ** (-snr_db / 20.0) / np.sqrt(2.0)
    )

    noise = noise_scale[:, None] * (
        rng.normal(size=symbols.shape)
        + 1j * rng.normal(size=symbols.shape)
    )

    waveforms = (
        gain[:, None]
        * np.exp(1j * phase[:, None])
        * symbols
        + noise
    )

    return {
        "waveforms": waveforms,
        "labels": labels,
        "snr_db": snr_db,
    }


def extract_q3_features(waveforms):
    waveforms = np.asarray(waveforms, dtype=complex)

    if (
        waveforms.ndim != 2
        or waveforms.shape[0] == 0
        or waveforms.shape[1] < 2
    ):
        raise ValueError(
            "Expected a nonempty (bursts, samples) array."
        )

    if not np.isfinite(waveforms).all():
        raise ValueError("Waveforms must be finite.")

    power = np.abs(waveforms) ** 2
    mean_power = power.mean(axis=1)

    if np.any(mean_power <= 0):
        raise ValueError("Each burst must have positive power.")

    return np.column_stack([
        np.abs(np.mean(waveforms**2, axis=1)) / mean_power,
        np.mean(power**2, axis=1) / mean_power**2,
    ])

## 3.1 Data and Receiver Evidence

### 3.1a Generate and Validate the Data

Generate training and test partitions using the supplied
development SNR range and separate random streams.

- Extract the two features and store each partition in q3_partitions.
- Check dimensions, finite values and valid labels.
- Report partition sizes and training class counts.
- Create q3_train_X, q3_train_y, q3_test_X and q3_test_y.

Each observation represents one complete burst. Use training data only
for the exploratory plots. Fit preprocessing and model parameters on
training data only.

In [ ]:
# TODO
# Generate the partitions, extract features, and test the data.

### 3.1b Inspect the Receiver's Evidence

Create one figure containing:

- I/Q scatter plots for one training burst from each modulation scheme.
- The two-dimensional training feature space, colored by modulation label.

Use equal I/Q axis scales. Do not standardize the features for this inspection.

In [ ]:
# TODO
# Visualize example training bursts and the training feature space.

## 3.2 Learning from Labeled Transmissions

### 3.2a Implement Multiclass Gaussian Recognition

Implement LDA and QDA using training observations only.

- Estimate empirical class priors and class means.
- For LDA, estimate the shared within-class covariance using denominator n.
- For QDA, estimate each class covariance using denominator n_k.
- Apply the fixed shrinkage below to the appropriate covariance estimate:

$$
\widehat{\Sigma}
=(1-\rho)S+\rho\frac{\operatorname{tr}(S)}{d}I,
\qquad \rho=0.01.
$$

Compute normalized log posterior probabilities using Cholesky-based
calculations and logsumexp.

Use your own implementation. Library LDA/QDA estimators are **not permitted**

> Derive the Gaussian class scores and normalized log posterior probabilities here.

In [ ]:
# TODO
# Implement Gaussian fitting, log posterior probabilities, and evaluation tools.

### 3.2b Fit and Compare the Models

Fit preprocessing and both classifiers using the 600 training bursts.

For each model, report training log loss and test accuracy, log loss
and multiclass Brier score.

Fit both models before examining test performance. Do not select or
retune a model using the test results.

Use ten equal-width confidence bins on [0, 1], including 1 in the final bin.
For nonempty bins, report counts, mean top-label confidence and accuracy.

Use the multiclass Brier convention

$$
\frac{1}{n}\sum_i\sum_k
\left(\widehat p_{ik}-\mathbf 1\{y_i=k\}\right)^2.
$$

In [ ]:
# TODO
# Fit preprocessing and both classifiers on training data only.
# Evaluate both fixed models on test data.
# Report the metrics and top-label reliability tables.

### 3.2c Visual Evidence

Create a three-panel figure containing:

1. LDA decision regions with validation observations.
2. QDA decision regions with validation observations.
3. Validation top-label reliability curves for both models.

Mark classification errors and use consistent class colors.
Interpret reliability curves alongside the confidence-bin counts.

In [ ]:
# TODO
# Plot LDA/QDA decision regions and validation reliability curves.

## 3. Analysis

Answer briefly using your numerical results and figures.

1. What does one point represent in an I/Q plot and in the feature-space
   plot? Explain the features' gain/phase invariance, their sensitivity
   to additive noise, and why Gaussian class distributions remain a
   modeling assumption.

2. Explain the LDA/QDA boundary geometry and compare their test results.
   Why can accuracy, log loss and Brier score assess performance differently?

3. Interpret the reliability curves alongside their bin counts.
   Does top-label reliability establish calibration of every class
   probability? Why would choosing or retuning a model using these
   results compromise the independence of the test evaluation?

**Your answers:**

1. ...
2. ...
3. ...

# 4. Discovering Hidden Machine Operating States

Helpful links:

1_ [EM Algorithm](https://www.columbia.edu/~mh2078/MachineLearningORFE/EM_Algorithm.pdf)

2_ [Gaussian mixture models](https://scikit-learn.org/stable/modules/mixture.html)

3_ [Implementing EM Algorithm from Scratch with Python](https://towardsdatascience.com/implementing-expectation-maximisation-algorithm-from-scratch-with-python-9ccb2c8521b3/)

4_ [A Primer on the EM Algorithm](https://towardsdatascience.com/a-primer-on-the-em-algorithm-7bd60e9813e/)

5_ [Gaussian Mixture Model Clearly Explained](https://towardsdatascience.com/gaussian-mixture-model-clearly-explained-115010f7d4cf/)

6_ [GMMs: from Theory to Implementation](https://towardsdatascience.com/gaussian-mixture-models-gmms-from-theory-to-implementation-4406c7fe9847/)



A vibration-monitoring system records two measurements per time window:
low-frequency and high-frequency log band power, in decibels.

Discover the machine's hidden operating states using a Gaussian mixture
model and EM. The supplied simulator assumes Gaussian measurements
conditional on each state. This is a simplified measurement model.

Complete the E-step and M-step, compare candidate component counts using
BIC, and inspect the discovered structure.

Do not use generating labels or known simulator parameters for fitting,
initialization or selection. Generating labels are reserved for the
post-selection audit. Library mixture estimators are **not permitted**.

In [ ]:
# PROVIDED — run without modification.

Q4_SEED = 42
Q4_FLOOR = 1e-3
Q4_COMPONENTS = tuple(range(1, 6))
Q4_RESTARTS = 3

Q4_FEATURE_NAMES = (
    "Low-frequency log band power (dB)",
    "High-frequency log band power (dB)",
)


def q4_generate(n, seed):
    rng = np.random.default_rng(seed)

    weights = np.array([0.40, 0.35, 0.25])

    means = np.array([
        [10.0, 10.0],
        [14.0, 14.0],
        [18.0, 10.0],
    ])

    covariances = np.array([
        [[0.80,  0.35], [ 0.35, 0.60]],
        [[0.65, -0.30], [-0.30, 0.90]],
        [[0.90,  0.20], [ 0.20, 0.55]],
    ])

    states = rng.choice(3, size=n, p=weights)
    X = np.empty((n, 2))

    for k in range(3):
        rows = states == k

        X[rows] = rng.multivariate_normal(
            means[k], covariances[k], size=rows.sum()
        )

    return X, states


q4_train_X, q4_train_state = q4_generate(
    600, np.random.SeedSequence([Q4_SEED, 400, 0])
)

q4_test_X, q4_test_state = q4_generate(
    600, np.random.SeedSequence([Q4_SEED, 400, 1])
)


def q4_floor_covariance(S, floor):
    values, vectors = linalg.eigh((S + S.T) / 2)

    return (
        vectors * np.maximum(values, floor)
    ) @ vectors.T


def q4_logpdf(X, means, covariances):
    n, d = X.shape
    result = np.empty((n, len(means)))

    for k, mean in enumerate(means):
        L = linalg.cholesky(covariances[k], lower=True)

        residual = linalg.solve_triangular(
            L, (X - mean).T, lower=True
        )

        result[:, k] = -0.5 * (
            d * np.log(2 * np.pi)
            + 2 * np.log(np.diag(L)).sum()
            + (residual**2).sum(axis=0)
        )

    return result


def q4_initial_model(X, K, rng):
    indices = [int(rng.integers(len(X)))]

    for _ in range(1, K):
        distances = (
            (
                X[:, None, :]
                - X[indices][None, :, :]
            ) ** 2
        ).sum(axis=2).min(axis=1)

        indices.append(
            int(rng.choice(
                len(X),
                p=distances / distances.sum(),
            ))
        )

    covariance = q4_floor_covariance(
        np.cov(X, rowvar=False, bias=True),
        Q4_FLOOR,
    )

    return {
        "weights": np.full(K, 1 / K),
        "means": X[indices].copy(),
        "covariances": np.repeat(
            covariance[None], K, axis=0
        ),
    }


def q4_fit(X, K, seed, max_iter=500, tol=1e-6):
    model = q4_initial_model(
        X, K, np.random.default_rng(seed)
    )

    responsibilities, log_density = q4_e_step(X, model)
    history = [float(log_density.mean())]
    converged = False

    for _ in range(max_iter):
        model = q4_m_step(
            X, responsibilities, floor=Q4_FLOOR
        )

        responsibilities, log_density = q4_e_step(
            X, model
        )

        current = float(log_density.mean())
        gain = current - history[-1]

        if not np.isfinite(current) or gain < -1e-8:
            raise FloatingPointError(
                "Nonfinite or decreasing likelihood."
            )

        history.append(current)

        if abs(gain) <= tol:
            converged = True
            break

    model.update(
        history=np.asarray(history),
        converged=converged,
    )

    return model


def q4_run_candidates(X):
    runs = {}
    best_models = {}
    status = []

    for K in Q4_COMPONENTS:
        for restart in range(Q4_RESTARTS):
            try:
                model = q4_fit(
                    X,
                    K,
                    np.random.SeedSequence(
                        [Q4_SEED, 401, K, restart]
                    ),
                )

                runs[K, restart] = model

                message = (
                    "converged"
                    if model["converged"]
                    else "iteration limit"
                )

            except (
                FloatingPointError,
                linalg.LinAlgError,
            ) as error:
                message = str(error)

            status.append({
                "K": K,
                "restart": restart,
                "status": message,
            })

        eligible = [
            model
            for (k, _), model in runs.items()
            if k == K and model["converged"]
        ]

        if eligible:
            best_models[K] = max(
                eligible,
                key=lambda model: model["history"][-1],
            )

    return runs, best_models, pd.DataFrame(status)


print("Training measurements:", q4_train_X.shape)
print("Test measurements:", q4_test_X.shape)

## 4.1 Implement and Check EM

Derive the responsibilities and weighted updates for mixing weights,
means and covariances. Write your derivation below.

Complete the two functions in the next cell.

- q4_e_step returns responsibilities of shape (n, K) and
  per-observation log densities of shape (n,).
- q4_m_step returns a dictionary containing weights, means and covariances.
- Use q4_logpdf and logsumexp for stable responsibilities.
- Apply q4_floor_covariance to each updated covariance.
- Report a numerical failure if a component's effective count is negligible.

Then run the supplied one-iteration check. Its covariance floor is zero;
the main experiment uses Q4_FLOOR.

In [ ]:
# TODO — implement the E-step and M-step.

def q4_e_step(X, model):
    """
    Return:
    responsibilities : array, shape (n, K)
    log_density : array, shape (n,)
    """
    raise NotImplementedError


def q4_m_step(X, responsibilities, floor=Q4_FLOOR):
    """
    Return:
        weights: shape (K,)
        means: shape (K, d)
        covariances: shape (K, d, d)
    """
    raise NotImplementedError

In [ ]:
# PROVIDED — verify one complete E/M iteration.

q4_toy_X = np.array(
    [-2.0, -1.0, 0.0, 3.0, 4.0]
)[:, None]

q4_toy_model = {
    "weights": np.array([0.5, 0.5]),
    "means": np.array([[-1.0], [3.0]]),
    "covariances": np.ones((2, 1, 1)),
}

q4_toy_r, q4_toy_before = q4_e_step(
    q4_toy_X, q4_toy_model
)

q4_toy_updated = q4_m_step(
    q4_toy_X, q4_toy_r, floor=0.0
)

_, q4_toy_after = q4_e_step(
    q4_toy_X, q4_toy_updated
)

np.testing.assert_allclose(
    q4_toy_r.sum(axis=1), 1.0
)
assert np.all(q4_toy_r >= 0)

np.testing.assert_allclose(
    q4_toy_updated["weights"],
    [0.596403, 0.403597],
    atol=1e-5,
)

np.testing.assert_allclose(
    q4_toy_updated["means"][:, 0],
    [-1.005569, 3.468122],
    atol=1e-5,
)

np.testing.assert_allclose(
    q4_toy_updated["covariances"][:, 0, 0],
    [0.666474, 0.359354],
    atol=1e-5,
)

assert (
    q4_toy_after.sum()
    >= q4_toy_before.sum() - 1e-10
)

print(pd.DataFrame({
    "effective_count": q4_toy_r.sum(axis=0),
    "weight": q4_toy_updated["weights"],
    "mean": q4_toy_updated["means"][:, 0],
    "variance": q4_toy_updated["covariances"][:, 0, 0],
}).to_string(index=False))

print(
    "Total log likelihood:",
    f"{q4_toy_before.sum():.6f}",
    "->",
    f"{q4_toy_after.sum():.6f}",
)

## 4.2 Discover the Operating States

Fit standardization on q4_train_X only. Transform training and test
measurements using that same transformation.

Use q4_run_candidates to fit K = 1, ..., 5 with three restarts each.
Inspect its restart-status table. Every candidate must have at least one
converged run before completing the comparison.

For each retained model, report:

- Train and test mean log density in standardized coordinates.
- Number of free parameters.
- Training BIC.

For d-dimensional full-covariance mixtures,

$$
q_K=(K-1)+Kd+K\frac{d(d+1)}{2},
\qquad
BIC=-2\ell_{train}+q_K\log n.
$$

Select the smallest training BIC. Record q4_selected_K and q4_selected_model
before accessing state labels.

In [ ]:
# TODO

# 1. Create q4_scaler, q4_train_Z and q4_test_Z.
# 2. Obtain q4_runs, q4_best_models and q4_restart_status.
# 3. Print restart diagnostics and check candidate completeness.
# 4. Build and print q4_comparison with:
#    K, parameters, train_mean_log_density,
#    test_mean_log_density, bic.
# 5. Define q4_selected_K and q4_selected_model.

## 4.3 Inspect and Interpret

After fixing the selected model:

- Report test ARI and a generating-state versus fitted-component
  contingency table.
- Find the test observation with the lowest maximum responsibility.
  Report its original measurements and responsibility vector.
- Create one two-panel figure: restart likelihood trajectories for the
  selected K, and training measurements colored by fitted component with
  95% Gaussian component probability contours in original coordinates.

Component IDs are arbitrary. Contours describe component probability;
they are not cluster boundaries.


In [ ]:
# TODO

# 1. Compute test responsibilities and assignments.
# 2. Report ARI and the state–component contingency table.
# 3. Report the most ambiguous test observation.
# 4. Produce the required two-panel figure.

# Use q4_test_state only for the post-selection audit.
# Convert fitted means and covariances back to original coordinates
# before drawing the component contours.

## 4. Analysis

Answer briefly using your numerical results and figures.

1. Does the selected mixture recover the generating states?
   Use BIC, restart trajectories and the post-selection audit.
   Why does increasing likelihood during EM not guarantee a global optimum?

2. Interpret the responsibility vector of the selected ambiguous observation.
   Does a large maximum responsibility establish that an observation
   lies in a familiar, high-density region?

3. Can a discovered component be identified as a fault state without
   additional information? Explain.

**Your answers:**

1. ...
2. ...
3. ...